In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install Sastrawi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import re

# 1. Memuat data hasil Tugas 2
df = pd.read_csv('/content/drive/MyDrive/Muhammad Yahya Ayyasy_Tugas 21 Agustus 2026/data_labeled.csv')

print('Dimensi data:', df.shape)
print('\n5 baris pertama:')
print(df.head())
print('\nDistribusi sentimen:')
print(df['sentimen'].value_counts())

Dimensi data: (7948, 6)

5 baris pertama:
                                     id  \
0  703e299d-a065-4520-bdc2-cdaf7072d29b   
1  c634c017-5c27-41e6-9051-802645e00c07   
2  ac27ad4e-b8a8-48c2-87e9-f47fd94b65a8   
3  47e6833b-3e3d-42a6-b48f-6f2a7fdef41a   
4  faec6ad2-1379-41af-a0a1-ec8440fcb085   

                                                teks              tanggal  \
0  akun sudah terverifikasi atau sudah premium, t...  2026-08-01 08:02:57   
1  saya senang dengan caranya yang mudah, tetapi ...  2026-08-14 06:41:52   
2  kurang bagus, karna saya login , tidak bisa ma...  2026-08-16 08:00:05   
3  kecewa beraatttt sebagai pelanggan aplikasi da...  2026-08-17 06:51:19   
4  penjagaan dana yang tersedia sangat longgar, s...  2026-08-21 17:20:28   

      sumber                                        teks_normal sentimen  
0  playstore  akun sudah terverifikasi atau sudah premium ta...  negatif  
1  playstore  saya senang dengan caranya yang mudah tetapi k...  negatif  
2  playstor

In [ ]:
# 2. Fungsi Text Cleaning dengan Regex
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()                                  # case folding
    text = re.sub(r'https?://\S+|www\.\S+', '', text)    # hapus URL
    text = re.sub(r'@\w+|#\w+', '', text)                # hapus mention/hashtag
    text = re.sub(r'\d+', '', text)                      # hapus angka
    text = re.sub(r'[^\w\s]', '', text)                  # hapus tanda baca
    text = re.sub(r'\s+', ' ', text).strip()             # spasi berlebih
    return text

df['teks_clean'] = df['teks'].apply(clean_text)
print("\nContoh Hasil Cleaning:")
print(df[['teks', 'teks_clean']].head())


Contoh Hasil Cleaning:
                                                teks  \
0  akun sudah terverifikasi atau sudah premium, t...   
1  saya senang dengan caranya yang mudah, tetapi ...   
2  kurang bagus, karna saya login , tidak bisa ma...   
3  kecewa beraatttt sebagai pelanggan aplikasi da...   
4  penjagaan dana yang tersedia sangat longgar, s...   

                                          teks_clean  
0  akun sudah terverifikasi atau sudah premium ta...  
1  saya senang dengan caranya yang mudah tetapi k...  
2  kurang bagus karna saya login tidak bisa masuk...  
3  kecewa beraatttt sebagai pelanggan aplikasi da...  
4  penjagaan dana yang tersedia sangat longgar sa...  


In [ ]:
# 3. Tokenisasi, Stop Word Removal, dan Stemming dengan Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stemmer = StemmerFactory().create_stemmer()
stopword = StopWordRemoverFactory().create_stop_word_remover()

def preprocess_full(text):
    text = clean_text(text)              # cleaning
    text = stopword.remove(text)         # hapus stop word
    text = stemmer.stem(text)            # stemming
    return text

# Mengaplikasikan preprocessing (Proses ini memakan waktu beberapa menit untuk ribuan baris)
df['teks_processed'] = df['teks_clean'].apply(preprocess_full)

print("\nContoh Hasil Full Preprocessing:")
print(df[['teks_clean', 'teks_processed']].head())


Contoh Hasil Full Preprocessing:
                                          teks_clean  \
0  akun sudah terverifikasi atau sudah premium ta...   
1  saya senang dengan caranya yang mudah tetapi k...   
2  kurang bagus karna saya login tidak bisa masuk...   
3  kecewa beraatttt sebagai pelanggan aplikasi da...   
4  penjagaan dana yang tersedia sangat longgar sa...   

                                      teks_processed  
0  akun verifikasi sudah premium belum transfer e...  
1  senang cara mudah kenapa kalau kirim saldo isi...  
2  kurang bagus karna login bisa masuk alas ups j...  
3  kecewa beraatttt langgan aplikasi dana lama tm...  
4  jaga dana sedia sangat longgar sering colong l...  


In [ ]:
# 4. Simpan hasil
df.to_csv('data_preprocessed.csv', index=False)
print('\nDataset berhasil dipreprocessing dan disimpan sebagai data_preprocessed.csv')


Dataset berhasil dipreprocessing dan disimpan sebagai data_preprocessed.csv
